# F1 Podium Prediction: Milestone 1

**Goal:** for every driver in every race, predict whether they finish in the top 3 (the podium).

- **One row = one driver in one race**
- **What we predict:** podium = finishing position 1, 2 or 3
- **When we predict:** after qualifying, before the race starts. Only information known at that moment may be used.
- **Data:** Kaggle / Ergast Formula 1 dataset, 1950-2024 (CSV files in the `csvs/` folder)

This notebook assumes no prior knowledge of the data: everything we claim about it is checked with code first.

In [1]:
# --- Libraries ---
import pandas as pd                 # tables (DataFrames)
import numpy as np                  # numbers and arrays
import matplotlib.pyplot as plt     # plots
import seaborn as sns               # nicer statistical plots
from pathlib import Path            # file and folder paths

# --- Display settings ---
pd.set_option("display.max_columns", 50)   # show all columns of a table
sns.set_theme(style="whitegrid")           # clean plot style

In [2]:
# Folder with the CSV files
DATA_DIR = Path("csvs")

# Load every CSV as plain text
text = {}
for file in sorted(DATA_DIR.glob("*.csv")):
    text[file.stem] = pd.read_csv(file, dtype=str, keep_default_na=False)

# Rows and columns of each table
sizes = pd.DataFrame({name: df.shape for name, df in text.items()}, index=["rows", "columns"]).T
sizes

,rows,columns
circuits,77,9
constructor_results,12625,5
constructor_standings,13391,7
constructors,212,5
driver_standings,34863,7
drivers,861,9
lap_times,589081,6
pit_stops,11371,7
qualifying,10494,9
races,1125,18


In [3]:
# First look: the first 2 rows of every table
for name, df in text.items():
    print(f"=== {name} ({len(df):,} rows) ===")
    display(df.head(2))

=== circuits (77 rows) ===


,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,http://en.wikipedia.org/wiki/Melbourne_Grand_P...
1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,http://en.wikipedia.org/wiki/Sepang_Internatio...


=== constructor_results (12,625 rows) ===


,constructorResultsId,raceId,constructorId,points,status
0,1,18,1,14,\N
1,2,18,2,8,\N


=== constructor_standings (13,391 rows) ===


...

,year,url
0,2009,http://en.wikipedia.org/wiki/2009_Formula_One_...
1,2008,http://en.wikipedia.org/wiki/2008_Formula_One_...


=== sprint_results (360 rows) ===


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId
0,1,1061,830,9,33,2,1,1,1,3,17,25:38.426,1538426,14,1:30.013,1
1,2,1061,1,131,44,1,2,2,2,2,17,+1.430,1539856,17,1:29.937,1


=== status (139 rows) ===


,statusId,status
0,1,Finished
1,2,Disqualified


## The tables and how they connect
Based on the column names in the first look: tables share columns such as `raceId` or `driverId`. These links are checked with code later.

```
Level 0   seasons   circuits   drivers   constructors   status      (stand on their own)
              \       /           |            |           |
Level 1        races              |            |           |        (each race: one season, one circuit)
                 |                |            |           |
Level 2   results, qualifying, sprint_results, standings, lap_times, pit_stops
```

| Table | One row is | Connects to (through) |
|---|---|---|
| `seasons` | a year | – |
| `circuits` | a race track | – |
| `drivers` | a driver | – |
| `constructors` | a team | – |
| `status` | a reason a race ended | – |
| `races` | a Grand Prix | `seasons` (`year`), `circuits` (`circuitId`) |
| `results` | a driver's result in a race | `races` (`raceId`), `drivers` (`driverId`), `constructors` (`constructorId`), `status` (`statusId`) |
| `qualifying` | a driver's qualifying in a race | `races` (`raceId`), `drivers` (`driverId`), `constructors` (`constructorId`) |
| `sprint_results` | a driver's result in a sprint | `races` (`raceId`), `drivers` (`driverId`), `constructors` (`constructorId`), `status` (`statusId`) |
| `driver_standings` | a driver's championship points and rank at a race | `races` (`raceId`), `drivers` (`driverId`) |
| `constructor_standings` | a team's championship points and rank at a race | `races` (`raceId`), `constructors` (`constructorId`) |
| `constructor_results` | a team's points in a race | `races` (`raceId`), `constructors` (`constructorId`) |
| `lap_times` | one lap of a driver in a race | `races` (`raceId`), `drivers` (`driverId`) |
| `pit_stops` | one pit stop of a driver in a race | `races` (`raceId`), `drivers` (`driverId`) |

## Missing values, table by table
Files can write "no value" in different ways. For every column we count blank cells and the common placeholders (`\N`, `NA`, `NULL`, `None`, `nan`, `-`, `?`). Only the ones that are found are shown.

In [7]:
# Common ways a file writes "no value" (blank cells are counted separately)
PLACEHOLDERS = ["\\N", "NA", "N/A", "NULL", "null", "None", "nan", "NaN", "-", "?"]

def missing_report(name):
    """Count blank cells and placeholders in every column of a table."""
    df = text[name]
    counts = {"blank": (df.apply(lambda col: col.str.strip()) == "").sum()}
    for p in PLACEHOLDERS:
        counts[p] = (df == p).sum()
    report = pd.DataFrame(counts)
    report = report.loc[:, report.sum() > 0]                       # keep only what was found
    report["rows missing"] = report.sum(axis=1).astype(int)        # blank + placeholders
    report["% missing"] = (report["rows missing"] / len(df) * 100).round(1)
    print(f"{name}: {len(df):,} rows")
    return report

### `seasons`
One row per season (year). Columns: `year`, `url`. A season can have many races

In [8]:
missing_report("seasons")

seasons: 75 rows


,rows missing,% missing
year,0,0.0
url,0,0.0


**Result:** no blank cells and no placeholders. All 75 rows are complete.

### `circuits`
One row per race track. Columns: `circuitId`, `circuitRef`, `name`, `location`, `country`, `lat`, `lng`, `alt`, `url`. A circuit can host many races.

In [9]:
missing_report("circuits")

circuits: 77 rows


,rows missing,% missing
circuitId,0,0.0
circuitRef,0,0.0
name,0,0.0
location,0,0.0
country,0,0.0
lat,0,0.0
lng,0,0.0
alt,0,0.0
url,0,0.0


**Result:** no blank cells and no placeholders. All 77 rows are complete.

### `drivers`
One row per driver. Columns: `driverId`, `driverRef`, `number`, `code`, `forename`, `surname`, `dob`, `nationality`, `url`. A driver can take part in many races.

In [10]:
missing_report("drivers")

drivers: 861 rows


,\N,rows missing,% missing
driverId,0,0,0.0
driverRef,0,0,0.0
number,802,802,93.1
code,757,757,87.9
forename,0,0,0.0
surname,0,0,0.0
dob,0,0,0.0
nationality,0,0,0.0
url,0,0,0.0


**Result:** the only placeholder found is `\N`. Two columns have missing values:
- `number`: 802 of 861 drivers (93.1%)
- `code`: 757 of 861 drivers (87.9%)

All other columns are complete. This is the first table where `\N` shows up as the way "no value" is written.

**Are `number` and `code` important? No, so we do not fill them in.**
- `driverId` already identifies every driver, is complete, and is the column other tables use to link to drivers.
- `code` is only a 3-letter short form of the surname (e.g. `HAM` for Hamilton) and `number` is a car number: labels, not information about performance.
- Filling them would mean inventing values that were never recorded.

### `constructors`
One row per team. Columns: `constructorId`, `constructorRef`, `name`, `nationality`, `url`. A team can take part in many races.

In [11]:
missing_report("constructors")

constructors: 212 rows


,rows missing,% missing
constructorId,0,0.0
constructorRef,0,0.0
name,0,0.0
nationality,0,0.0
url,0,0.0


**Result:** no blank cells and no placeholders. All 212 rows are complete.

### `status`
One row per reason a race ended (e.g. `Finished`, `Disqualified`). Columns: `statusId`, `status`. The same reason can be used by many results.

In [12]:
missing_report("status")

status: 139 rows


,rows missing,% missing
statusId,0,0.0
status,0,0.0


**Result:** no blank cells and no placeholders. All 139 rows are complete.

### `races`
One row per Grand Prix. Columns: `raceId`, `year`, `round`, `circuitId`, `name`, `date`, `time`, `url`, and the date and time of practice sessions (`fp1`-`fp3`), qualifying (`quali`) and the sprint. A race belongs to one season and one circuit.

In [13]:
missing_report("races")

races: 1,125 rows


,\N,rows missing,% missing
raceId,0,0,0.0
year,0,0,0.0
round,0,0,0.0
circuitId,0,0,0.0
name,0,0,0.0
date,0,0,0.0
time,731,731,65.0
url,0,0,0.0
fp1_date,1035,1035,92.0
fp1_time,1057,1057,94.0


**Result:** the only placeholder found is `\N`. The main race columns (`raceId`, `year`, `round`, `circuitId`, `name`, `date`, `url`) are complete. Missing values:
- `time` (race start time): 731 of 1,125 races (65.0%)
- practice dates and times (`fp1`-`fp3`): 92.0% to 95.3%
- qualifying date and time: 92.0% / 94.0%
- sprint date and time: 98.4% / 98.7%

**Do we need these columns? No, so we do not deal with their missing values.**
- They are the **schedule** of the weekend (when each session took place), not results. They are known before the race, so they are not leakage, but they say nothing about how fast a driver or team is.
- They are empty for most races (65% to 98.7%, see above).
- Qualifying **results** are in the `qualifying` table, and sprint **results** are in `sprint_results`, so nothing useful is lost.

These columns will be dropped when cleaning. From `races` we keep `raceId`, `year`, `round`, `circuitId`, `name` and `date`.

### `results`
One row per driver in a race. Columns: `resultId`, `raceId`, `driverId`, `constructorId`, `number`, `grid`, `position`, `positionText`, `positionOrder`, `points`, `laps`, `time`, `milliseconds`, `fastestLap`, `rank`, `fastestLapTime`, `fastestLapSpeed`, `statusId`.

In [15]:
missing_report("results")

results: 26,759 rows


,\N,rows missing,% missing
resultId,0,0,0.0
raceId,0,0,0.0
driverId,0,0,0.0
constructorId,0,0,0.0
number,6,6,0.0
grid,0,0,0.0
position,10953,10953,40.9
positionText,0,0,0.0
positionOrder,0,0,0.0
points,0,0,0.0


**Result:** the only placeholder found is `\N`. The ID columns, `grid`, `positionText`, `positionOrder`, `points`, `laps` and `statusId` are complete. Missing values:
- `position`: 10,953 of 26,759 rows (40.9%), while `positionOrder` is never missing
- `time` and `milliseconds`: 19,079 rows (71.3%)
- fastest-lap columns (`fastestLap`, `rank`, `fastestLapTime`, `fastestLapSpeed`): about 69%
- `number`: only 6 rows (0.02%, shown as 0.0)

**Do we need to deal with these missing values? No.**
- `position`: empty when a driver has no official finishing place. `positionText` then holds a letter code instead (R retired, D disqualified, E excluded, W withdrawn, F failed to qualify, N not classified), and our target uses `positionOrder`, which is never missing.
- `time` and `milliseconds`: the finishing time or gap to the winner. For the race we are predicting this is **post-race** information (leakage). Values from **earlier** races would be allowed, but we do not use them: they are missing for 71% of rows (only drivers who finished the full race have one), and are written in different ways (the winner's total time or a gap to the winner).
- `fastestLap`, `rank`, `fastestLapTime`, `fastestLapSpeed`: the same rule. They are leakage for the race being predicted. Values from **earlier** races would be allowed, but they are missing for about 69% of rows (nothing is recorded before 2004), and a driver's recent form is already covered by past `positionOrder` and `points`, which exist for every race.
- `number`: the car number, only 6 rows, and it says nothing about performance.

**Columns we will use from `results`:**
- `raceId`, `driverId`, `constructorId`: to identify the driver and race, and to join other tables
- `grid`: the starting position, known before the race
- `positionOrder`: to build the target (podium = `positionOrder` 1, 2 or 3), and from past races for recent form
- `points`, `statusId`: only from **past** races (recent form, reliability), never from the race being predicted

### `qualifying`
One row per driver in a qualifying session. Columns: `qualifyId`, `raceId`, `driverId`, `constructorId`, `number`, `position`, `q1`, `q2`, `q3` (lap times in the three parts of qualifying).

In [16]:
missing_report("qualifying")

qualifying: 10,494 rows


,blank,\N,rows missing,% missing
qualifyId,0,0,0,0.0
raceId,0,0,0,0.0
driverId,0,0,0,0.0
constructorId,0,0,0,0.0
number,0,0,0,0.0
position,0,0,0,0.0
q1,0,156,156,1.5
q2,22,4625,4647,44.3
q3,46,6819,6865,65.4


**Result:** this is the first table with **two kinds** of missing value: `\N` **and blank cells**. Only the lap-time columns have gaps:
- `q1`: 156 rows (1.5%), all `\N`
- `q2`: 4,647 rows (44.3%): 4,625 `\N` + 22 blank
- `q3`: 6,865 rows (65.4%): 6,819 `\N` + 46 blank

All other columns (including `position`) are complete. Both kinds must be treated as missing when cleaning.

**What do we do with these missing values?**
- **Blank cells and `\N`** in `q2` / `q3` mean the same thing, so both are treated as missing.
- **`q1`, `q2`, `q3` are not filled in.** An empty `q2` / `q3` means the driver was knocked out in an earlier part of qualifying (or that part did not exist that season), which is itself information. Filling it with a made-up time would hide that. Grid position cannot fill them either: a lap time (seconds) and a position (rank) measure different things.
- Instead, the three times will be turned into **one number** when building features: the driver's **best qualifying time**, compared with the fastest driver in that race (using whichever of `q1` / `q2` / `q3` exists).
- **Races with no qualifying rows at all** are a different problem: there the qualifying **position** is missing for every driver. Options are to use `grid` instead, fill it in, or only use seasons with qualifying data. This is decided after exploring the data.

**Columns we will use from `qualifying`:** `raceId`, `driverId` (to join with `results`), `position` (qualifying rank) and `q1`, `q2`, `q3` (for the best qualifying time).

### `sprint_results`
One row per driver in a sprint (a short extra race). Same columns as `results` except `rank` and `fastestLapSpeed`: `resultId`, `raceId`, `driverId`, `constructorId`, `number`, `grid`, `position`, `positionText`, `positionOrder`, `points`, `laps`, `time`, `milliseconds`, `fastestLap`, `fastestLapTime`, `statusId`.

In [17]:
missing_report("sprint_results")

sprint_results: 360 rows


,\N,rows missing,% missing
resultId,0,0,0.0
raceId,0,0,0.0
driverId,0,0,0.0
constructorId,0,0,0.0
number,0,0,0.0
grid,0,0,0.0
position,15,15,4.2
positionText,0,0,0.0
positionOrder,0,0,0.0
points,0,0,0.0


**Result:** the only placeholder found is `\N`, and only a few rows are affected:
- `position`: 15 of 360 rows (4.2%)
- `time` and `milliseconds`: 20 rows (5.6%)
- `fastestLap` and `fastestLapTime`: 9 rows (2.5%)

All other columns are complete. The table is small: 360 rows in total.

**Do we need to deal with these missing values? No.** They are the same kind as in `results`: `position` is empty when a driver has no official place in the sprint (`positionText` holds a letter code, and `positionOrder` is never missing), and `time`, `milliseconds` and the fastest-lap columns describe what happened during the sprint, so they are not needed. Whether sprint results are used at all (only a small number of races have a sprint) is decided later; if they are, we only need `raceId`, `driverId`, `positionOrder` and `points`.

### `driver_standings`
One row per driver per race: the driver's championship points, rank and wins. Columns: `driverStandingsId`, `raceId`, `driverId`, `points`, `position`, `positionText`, `wins`.

In [18]:
missing_report("driver_standings")

driver_standings: 34,863 rows


,rows missing,% missing
driverStandingsId,0,0.0
raceId,0,0.0
driverId,0,0.0
points,0,0.0
position,0,0.0
positionText,0,0.0
wins,0,0.0


**Result:** no blank cells and no placeholders. All 34,863 rows are complete.

### `constructor_standings`
One row per team per race: the team's championship points, rank and wins. Columns: `constructorStandingsId`, `raceId`, `constructorId`, `points`, `position`, `positionText`, `wins`.

In [19]:
missing_report("constructor_standings")

constructor_standings: 13,391 rows


,rows missing,% missing
constructorStandingsId,0,0.0
raceId,0,0.0
constructorId,0,0.0
points,0,0.0
position,0,0.0
positionText,0,0.0
wins,0,0.0


**Result:** no blank cells and no placeholders. All 13,391 rows are complete.

### `constructor_results`
One row per team per race: the points the team scored in that race. Columns: `constructorResultsId`, `raceId`, `constructorId`, `points`, `status`.

In [20]:
missing_report("constructor_results")

constructor_results: 12,625 rows


,\N,rows missing,% missing
constructorResultsId,0,0,0.0
raceId,0,0,0.0
constructorId,0,0,0.0
points,0,0,0.0
status,12608,12608,99.9


**Result:** the only placeholder found is `\N`, in one column: `status` is missing in 12,608 of 12,625 rows (99.9%), so it is filled in only 17 rows. `raceId`, `constructorId` and `points` are complete.

**How do we deal with the missing `status`? We drop the column.** It is filled in only 17 of 12,625 rows (with `D`, disqualified), so it carries almost no information, and a team's race outcome is already available per driver in `results` (`statusId`).

### `lap_times`
One row per lap of a driver in a race. Columns: `raceId`, `driverId`, `lap`, `position`, `time`, `milliseconds`.

In [21]:
missing_report("lap_times")

lap_times: 589,081 rows


,rows missing,% missing
raceId,0,0.0
driverId,0,0.0
lap,0,0.0
position,0,0.0
time,0,0.0
milliseconds,0,0.0


**Result:** no blank cells and no placeholders. All 589,081 rows are complete.

### `pit_stops`
One row per pit stop of a driver in a race. Columns: `raceId`, `driverId`, `stop`, `lap`, `time`, `duration`, `milliseconds`.

In [22]:
missing_report("pit_stops")

pit_stops: 11,371 rows


,rows missing,% missing
raceId,0,0.0
driverId,0,0.0
stop,0,0.0
lap,0,0.0
time,0,0.0
duration,0,0.0
milliseconds,0,0.0


**Result:** no blank cells and no placeholders. All 11,371 rows are complete.

## Formats and data types, table by table
All values are still text. These are the formats every column will be converted to:

| Kind of column | Examples | Format |
|---|---|---|
| IDs and counts | `raceId`, `grid`, `laps`, `positionOrder` | whole number |
| Points and coordinates | `points`, `lat`, `lng` | decimal number |
| Dates | `date`, `dob` | date (`YYYY-MM-DD`) |
| Lap times and durations | `q1`, `q2`, `q3`, pit-stop `duration` | seconds (decimal number) |
| Names and labels | `name`, `nationality`, `status` | text |

The code below says how each column is written: **number**, **date**, **time**, **text**, or **mixed** (written in more than one way). For mixed columns it lists the formats with how many rows use each (`num` = any number, e.g. `1:26.572` → `num:num.num`).

In [29]:
def written_as(values):
    """Say how a column is written: number, date, time, text, or mixed."""
    if pd.to_numeric(values, errors="coerce").notna().all():
        return "number"
    if values.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all():          # e.g. 2009-03-29
        return "date"
    if values.str.fullmatch(r"\d+(:\d+)+(\.\d+)?").all():          # e.g. 1:26.572 or 17:05:23
        return "time"
    if values.str.contains(r"[A-Za-z]").mean() > 0.5:              # mostly words: names, links
        return "text"
    return "mixed"

def format_report(name):
    """For every column: how it is written, an example, and its formats if it is mixed."""
    df = text[name]
    rows = {}
    for col in df.columns:
        values = df[col][~df[col].isin(["\\N", ""])]          # skip missing values
        kind = written_as(values)
        formats = ""
        if kind == "mixed":                                   # list each format with its count (num = any number)
            counts = values.str.replace(r"\d+", "num", regex=True).value_counts()
            formats = ", ".join(f"{f} ({n})" for f, n in counts.items())
        rows[col] = {"written as": kind, "example": values.iloc[0], "formats (if mixed)": formats}
    print(f"{name}: {len(df):,} rows")
    return pd.DataFrame(rows).T

### `seasons`

In [30]:
format_report("seasons")

seasons: 75 rows


,written as,example,formats (if mixed)
year,number,2009,
url,text,http://en.wikipedia.org/wiki/2009_Formula_One_...,


**Result:** `year` is written as a number, so it becomes a whole number. `url` is text (a link) and is not needed.

### `circuits`

In [31]:
format_report("circuits")

circuits: 77 rows


,written as,example,formats (if mixed)
circuitId,number,1,
circuitRef,text,albert_park,
name,text,Albert Park Grand Prix Circuit,
location,text,Melbourne,
country,text,Australia,
lat,number,-37.8497,
lng,number,144.968,
alt,number,10,
url,text,http://en.wikipedia.org/wiki/Melbourne_Grand_P...,


**Result:** no mixed columns. `circuitId` and `alt` become whole numbers, `lat` and `lng` decimal numbers; `circuitRef`, `name`, `location`, `country` and `url` stay text.

### `drivers`

In [32]:
format_report("drivers")

drivers: 861 rows


,written as,example,formats (if mixed)
driverId,number,1,
driverRef,text,hamilton,
number,number,44,
code,text,HAM,
forename,text,Lewis,
surname,text,Hamilton,
dob,date,1985-01-07,
nationality,text,British,
url,text,http://en.wikipedia.org/wiki/Lewis_Hamilton,


**Result:** no mixed columns. `driverId` becomes a whole number and `dob` (date of birth) a date; the names, `code`, `nationality` and `url` stay text. (`number` is a number but is not used, see the missing values.)

### `constructors`

In [33]:
format_report("constructors")

constructors: 212 rows


,written as,example,formats (if mixed)
constructorId,number,1,
constructorRef,text,mclaren,
name,text,McLaren,
nationality,text,British,
url,text,http://en.wikipedia.org/wiki/McLaren,


**Result:** no mixed columns. `constructorId` becomes a whole number; `constructorRef`, `name`, `nationality` and `url` stay text.

### `status`

In [34]:
format_report("status")

status: 139 rows


,written as,example,formats (if mixed)
statusId,number,1,
status,text,Finished,


**Result:** no mixed columns. `statusId` becomes a whole number; `status` stays text.

### `races`

In [35]:
format_report("races")

races: 1,125 rows


,written as,example,formats (if mixed)
raceId,number,1,
year,number,2009,
round,number,1,
circuitId,number,1,
name,text,Australian Grand Prix,
date,date,2009-03-29,
time,time,06:00:00,
url,text,http://en.wikipedia.org/wiki/2009_Australian_G...,
fp1_date,date,2021-04-16,
fp1_time,time,12:00:00,


**Result:** no mixed columns. `raceId`, `year`, `round` and `circuitId` become whole numbers and `date` a date; `name` stays text. The session dates and times are also written consistently, but they are dropped (see the missing values).

### `results`

In [36]:
format_report("results")

results: 26,759 rows


,written as,example,formats (if mixed)
resultId,number,1,
raceId,number,18,
driverId,number,1,
constructorId,number,1,
number,number,22,
grid,number,1,
position,number,1,
positionText,mixed,1,"num (15808), R (8897), F (1368), W (336), N (1..."
positionOrder,number,1,
points,number,10,


**Result:** two mixed columns:
- `positionText`: a number, or a letter code (`R`, `F`, `W`, `N`, `D`, `E`) when there is no finishing place. It stays text; we use `positionOrder` (a number) for the target.
- `time`: the winner's total time (`num:num:num.num`) or a gap to the winner (`+num.num`, `+num:num.num`). It is post-race information and is not used, so it does not need converting.

All the columns we keep (`raceId`, `driverId`, `constructorId`, `grid`, `positionOrder`, `points`, `statusId`) are numbers and become whole numbers (`points` a decimal number).

### `qualifying`

In [37]:
format_report("qualifying")

qualifying: 10,494 rows


,written as,example,formats (if mixed)
qualifyId,number,1,
raceId,number,18,
driverId,number,1,
constructorId,number,1,
number,number,22,
position,number,1,
q1,time,1:26.572,
q2,time,1:25.187,
q3,time,1:26.714,


**Result:** no mixed columns. The IDs, `number` and `position` become whole numbers. `q1`, `q2` and `q3` are all written as times like `1:26.572` (minutes:seconds), so they can be converted to seconds with one rule (e.g. `1:26.572` → 86.572).

### `sprint_results`

In [38]:
format_report("sprint_results")

sprint_results: 360 rows


,written as,example,formats (if mixed)
resultId,number,1,
raceId,number,1061,
driverId,number,830,
constructorId,number,9,
number,number,33,
grid,number,2,
position,number,1,
positionText,mixed,1,"num (345), R (13), N (1), W (1)"
positionOrder,number,1,
points,number,3,


**Result:** the same two mixed columns as in `results`: `positionText` (number or letter code) and `time` (total time or gap). Neither is needed. The columns we might keep (`raceId`, `driverId`, `positionOrder`, `points`) are numbers.

### `driver_standings`

In [39]:
format_report("driver_standings")

driver_standings: 34,863 rows


,written as,example,formats (if mixed)
driverStandingsId,number,1,
raceId,number,18,
driverId,number,1,
points,number,10,
position,number,1,
positionText,mixed,1,"num (34862), D (1)"
wins,number,1,


**Result:** one mixed column: `positionText` is a number in 34,862 rows and the letter `D` in 1 row. We use `position` (always a number) instead, so `positionText` is not needed. `points` becomes a decimal number; the IDs, `position` and `wins` become whole numbers.

### `constructor_standings`

In [40]:
format_report("constructor_standings")

constructor_standings: 13,391 rows


,written as,example,formats (if mixed)
constructorStandingsId,number,1,
raceId,number,18,
constructorId,number,1,
points,number,14,
position,number,1,
positionText,mixed,1,"num (13374), E (17)"
wins,number,1,


**Result:** one mixed column: `positionText` is a number in 13,374 rows and the letter `E` in 17 rows. As for drivers, we use `position` (always a number) instead. `points` becomes a decimal number; the IDs, `position` and `wins` become whole numbers.

### `constructor_results`

In [41]:
format_report("constructor_results")

constructor_results: 12,625 rows


,written as,example,formats (if mixed)
constructorResultsId,number,1,
raceId,number,18,
constructorId,number,1,
points,number,14,
status,text,D,


**Result:** no mixed columns. The IDs become whole numbers and `points` a decimal number. `status` is text, but it is dropped (see the missing values).

### `lap_times`

In [42]:
format_report("lap_times")

lap_times: 589,081 rows


,written as,example,formats (if mixed)
raceId,number,841,
driverId,number,20,
lap,number,1,
position,number,1,
time,time,1:38.109,
milliseconds,number,98109,


**Result:** no mixed columns. The IDs, `lap`, `position` and `milliseconds` become whole numbers; `time` is a time. This table is race-day detail (post-race), so it is not used as a feature.

### `pit_stops`

In [43]:
format_report("pit_stops")

pit_stops: 11,371 rows


,written as,example,formats (if mixed)
raceId,number,841,
driverId,number,153,
stop,number,1,
lap,number,1,
time,time,17:05:23,
duration,mixed,26.898,"num.num (10854), num:num.num (517)"
milliseconds,number,26898,


**Result:** one mixed column: `duration` is written in seconds (`26.898`, 10,854 rows) or in minutes:seconds (`16:44.718`, 517 rows). `milliseconds` holds the same duration as one number, so it can be used instead if needed. This table is race-day detail (post-race), so it is not used as a feature.

## Keys and links
- A **primary key** is the column (or columns) that identifies each row of a table, e.g. `raceId` in `races`. It must be **unique** (no value twice) and **never empty**. `lap_times` and `pit_stops` have no ID column of their own, so their key is a combination: `raceId` + `driverId` + `lap` (or `stop`).
- A **foreign key** is a column that points to a row in another table, e.g. `results.driverId` → `drivers.driverId`. Every value must exist in the table it points to; a value that points to nothing is a **broken link**.

In [44]:
# Primary key of each table: the column(s) that identify every row
PRIMARY_KEYS = {
    "seasons":               ["year"],
    "circuits":              ["circuitId"],
    "drivers":               ["driverId"],
    "constructors":          ["constructorId"],
    "status":                ["statusId"],
    "races":                 ["raceId"],
    "results":               ["resultId"],
    "qualifying":            ["qualifyId"],
    "sprint_results":        ["resultId"],
    "driver_standings":      ["driverStandingsId"],
    "constructor_standings": ["constructorStandingsId"],
    "constructor_results":   ["constructorResultsId"],
    "lap_times":             ["raceId", "driverId", "lap"],
    "pit_stops":             ["raceId", "driverId", "stop"],
}

# Check each key: no repeated values and no empty values
rows = {}
for table, key in PRIMARY_KEYS.items():
    df = text[table]
    rows[table] = {
        "primary key": " + ".join(key),
        "rows":        len(df),
        "repeated":    int(df.duplicated(subset=key).sum()),             # rows whose key already appeared
        "empty":       int(df[key].isin(["\\N", ""]).any(axis=1).sum()), # rows with an empty key
    }

pk_check = pd.DataFrame(rows).T
pk_check["valid"] = (pk_check["repeated"] == 0) & (pk_check["empty"] == 0)
pk_check

,primary key,rows,repeated,empty,valid
seasons,year,75,0,0,True
circuits,circuitId,77,0,0,True
drivers,driverId,861,0,0,True
constructors,constructorId,212,0,0,True
status,statusId,139,0,0,True
races,raceId,1125,0,0,True
results,resultId,26759,0,0,True
qualifying,qualifyId,10494,0,0,True
sprint_results,resultId,360,0,0,True
driver_standings,driverStandingsId,34863,0,0,True


**Result:** all 14 primary keys are valid: no repeated values and no empty values. Every row of every table can be identified.

Note: in `results` the key is `resultId`, not the combination `raceId` + `driverId`. Whether a driver appears more than once in the same race is checked in the duplicates part.

In [45]:
# Foreign keys: (table, column, table it points to). The column has the same name as the other table's primary key.
FOREIGN_KEYS = [
    ("races", "year", "seasons"),                   ("races", "circuitId", "circuits"),
    ("results", "raceId", "races"),                 ("results", "driverId", "drivers"),
    ("results", "constructorId", "constructors"),   ("results", "statusId", "status"),
    ("qualifying", "raceId", "races"),              ("qualifying", "driverId", "drivers"),
    ("qualifying", "constructorId", "constructors"),
    ("sprint_results", "raceId", "races"),          ("sprint_results", "driverId", "drivers"),
    ("sprint_results", "constructorId", "constructors"), ("sprint_results", "statusId", "status"),
    ("driver_standings", "raceId", "races"),        ("driver_standings", "driverId", "drivers"),
    ("constructor_standings", "raceId", "races"),   ("constructor_standings", "constructorId", "constructors"),
    ("constructor_results", "raceId", "races"),     ("constructor_results", "constructorId", "constructors"),
    ("lap_times", "raceId", "races"),               ("lap_times", "driverId", "drivers"),
    ("pit_stops", "raceId", "races"),               ("pit_stops", "driverId", "drivers"),
]

# Check each link: no empty values and every value exists in the table it points to
rows = []
for table, col, parent in FOREIGN_KEYS:
    values = text[table][col]
    rows.append({
        "link":         f"{table}.{col} -> {parent}",
        "empty":        int(values.isin(["\\N", ""]).sum()),
        "broken links": int((~values.isin(text[parent][col])).sum()),   # values that point to a row that does not exist
    })

fk_check = pd.DataFrame(rows)
fk_check["valid"] = (fk_check["empty"] == 0) & (fk_check["broken links"] == 0)
print(len(fk_check), "links checked,", int(fk_check["valid"].sum()), "valid")
fk_check

23 links checked, 23 valid


,link,empty,broken links,valid
0,races.year -> seasons,0,0,True
1,races.circuitId -> circuits,0,0,True
2,results.raceId -> races,0,0,True
3,results.driverId -> drivers,0,0,True
4,results.constructorId -> constructors,0,0,True
5,results.statusId -> status,0,0,True
6,qualifying.raceId -> races,0,0,True
7,qualifying.driverId -> drivers,0,0,True
8,qualifying.constructorId -> constructors,0,0,True
9,sprint_results.raceId -> races,0,0,True


**Result:** all 23 links are valid: no empty values and no broken links. Every race belongs to an existing season and circuit, and every result, qualifying, sprint, standing, lap and pit-stop row points to an existing race, driver, team and status. This confirms the connections described in "The tables and how they connect", so the tables can be joined safely without losing rows.

### Diagram of the relationships
`races` is in the centre. It points to `seasons` and `circuits` on the left, and every table in the middle column points to `races` through `raceId`. The tables on the right (`drivers`, `constructors`, `status`) say who took part and why a race ended. Each line is one of the 23 links checked above, and its colour shows the column that links the two tables.

![How the 14 tables are connected](F1_relationships.svg)

`drivers`, `constructors` and `status` are not linked to each other directly; they only meet inside the tables that describe a driver in a race.

### All relationships between the tables
All 23 links checked above are **one-to-many**: one row in the first table is linked to many rows in the second.

| From | To | Through | Relationship |
|---|---|---|---|
| `seasons` | `races` | `year` | one season has many races |
| `circuits` | `races` | `circuitId` | one circuit hosts many races (over the years) |
| `races` | `results` | `raceId` | one race has a result for every driver in it |
| `races` | `qualifying` | `raceId` | one race has a qualifying row for every driver |
| `races` | `sprint_results` | `raceId` | one race (sprint weekends only) has a sprint result for every driver |
| `races` | `driver_standings` | `raceId` | one race has a championship row for every driver after it |
| `races` | `constructor_standings` | `raceId` | one race has a championship row for every team after it |
| `races` | `constructor_results` | `raceId` | one race has a points row for every team in it |
| `races` | `lap_times` | `raceId` | one race has a row for every lap of every driver |
| `races` | `pit_stops` | `raceId` | one race has a row for every pit stop |
| `drivers` | `results` | `driverId` | one driver has a result in many races |
| `drivers` | `qualifying` | `driverId` | one driver has qualified in many races |
| `drivers` | `sprint_results` | `driverId` | one driver has many sprint results |
| `drivers` | `driver_standings` | `driverId` | one driver has a championship row after many races |
| `drivers` | `lap_times` | `driverId` | one driver has many lap times |
| `drivers` | `pit_stops` | `driverId` | one driver has many pit stops |
| `constructors` | `results` | `constructorId` | one team has many results (its drivers in many races) |
| `constructors` | `qualifying` | `constructorId` | one team has many qualifying rows |
| `constructors` | `sprint_results` | `constructorId` | one team has many sprint results |
| `constructors` | `constructor_standings` | `constructorId` | one team has a championship row after many races |
| `constructors` | `constructor_results` | `constructorId` | one team has a points row in many races |
| `status` | `results` | `statusId` | one reason (e.g. `Engine`) is used by many results |
| `status` | `sprint_results` | `statusId` | one reason is used by many sprint results |

**In short:** `races` is the centre, linked to `seasons` and `circuits` above it and to every result-type table below it. `drivers`, `constructors` and `status` describe who took part and why a race ended. `results`, `qualifying` and `sprint_results` share `raceId` + `driverId`, so they can be joined directly for the same driver in the same race.

## Duplicates
Our model needs **exactly one row per driver per race**, because each row is one prediction. Two checks:
1. **Identical rows:** the same row written twice.
2. **The same driver twice in one race:** two rows with the same `raceId` + `driverId`. The primary key (checked above) is a row ID, so it does not stop this.

### Check 1: identical rows

In [47]:
# Rows that are exactly the same as another row, in every table
pd.DataFrame({table: {"rows": len(df), "identical rows": int(df.duplicated().sum())}
              for table, df in text.items()}).T

,rows,identical rows
circuits,77,0
constructor_results,12625,0
constructor_standings,13391,0
constructors,212,0
driver_standings,34863,0
drivers,861,0
lap_times,589081,0
pit_stops,11371,0
qualifying,10494,0
races,1125,0


**Result:** no table contains an identical row.

### Check 2: the same driver (or team) twice in one race
A race should have one row per driver in the driver tables, and one row per team in the team tables.

In [49]:
# Tables with one row per driver per race, and tables with one row per team per race
per_driver = ["results", "qualifying", "sprint_results", "driver_standings"]
per_team = ["constructor_standings", "constructor_results"]

rows = {}
for table in per_driver + per_team:
    who = "driverId" if table in per_driver else "constructorId"
    pair_size = text[table].groupby(["raceId", who]).size()     # how many rows each driver-race (or team-race) pair has
    rows[table] = {
        "should be one row per":        "driver per race" if table in per_driver else "team per race",
        "pairs in total":               len(pair_size),
        "pairs with more than one row": int((pair_size > 1).sum()),
        "rows in those pairs":          int(pair_size[pair_size > 1].sum()),
    }

pd.DataFrame(rows).T

,should be one row per,pairs in total,pairs with more than one row,rows in those pairs
results,driver per race,26668,85,176
qualifying,driver per race,10494,0,0
sprint_results,driver per race,360,0,0
driver_standings,driver per race,34863,0,0
constructor_standings,team per race,13391,0,0
constructor_results,team per race,12625,0,0


**Result:** `qualifying`, `sprint_results`, `driver_standings`, `constructor_standings` and `constructor_results` have exactly one row per driver (or team) per race. **`results` does not:** 85 of its 26,668 driver-race pairs have more than one row (176 rows in total).

So `results` does **not** have one row per driver per race, and this must be fixed before it can be our modeling table. The next step is to look at these rows.

### Look at the repeated pairs in `results`
To read them easily we join `results` with `races` (year and race name), `drivers` (surname) and `status` (reason the race ended). This is the first join in the notebook. All three links were checked above, so no rows are lost.

In [50]:
# Join results with races, drivers and status
results = text["results"].copy()
for col in ["resultId", "grid", "positionOrder", "laps"]:
    results[col] = results[col].astype(int)

info = (results.merge(text["races"][["raceId", "year", "name"]], on="raceId")
               .merge(text["drivers"][["driverId", "surname"]], on="driverId")
               .merge(text["status"].rename(columns={"status": "status_text"}), on="statusId"))
info["year"] = info["year"].astype(int)
print("rows after the joins:", len(info), "(results had", len(results), "rows)")

# Keep only the rows whose driver-race pair appears more than once
repeated = info[info.duplicated(["raceId", "driverId"], keep=False)].sort_values(["raceId", "driverId", "resultId"])
pairs = repeated.groupby(["raceId", "driverId"])

print("rows per pair:", pairs.size().value_counts().to_dict())
pairs["year"].first().value_counts().sort_index().rename("repeated pairs").rename_axis("year").to_frame()

rows after the joins: 26759 (results had 26759 rows)
rows per pair: {2: 79, 3: 6}


,repeated pairs
year,
1950,4
1951,4
1952,3
1953,14
1954,16
1955,15
1956,12
1957,8
1958,3


**Result:** the joins keep all 26,759 rows. The 85 repeated pairs have 2 rows each (79 pairs) or 3 rows (6 pairs). They are all in early seasons: 1950 to 1964, plus one pair in 1978. After 1964 (except that one case) every driver has exactly one row per race.

In [51]:
# Examples: the repeated pairs from 1960 to 1964, one pair after another
columns = ["raceId", "year", "name", "surname", "constructorId", "grid", "positionOrder", "laps", "status_text"]
repeated[repeated["year"].between(1960, 1964)][columns]

,raceId,year,name,surname,constructorId,grid,positionOrder,laps,status_text
17362,717,1964,United States Grand Prix,Clark,172,1,7,102,Out of fuel
24298,717,1964,United States Grand Prix,Clark,172,1,12,54,Injection
17740,733,1962,British Grand Prix,Greene,172,0,22,0,Withdrew
17743,733,1962,British Grand Prix,Greene,97,0,25,0,Withdrew
17961,742,1961,British Grand Prix,Moss,102,20,19,56,Disqualified
17963,742,1961,British Grand Prix,Moss,172,5,21,44,Brakes
18062,745,1961,United States Grand Prix,Gregory,172,11,17,23,Gearbox
24297,745,1961,United States Grand Prix,Gregory,172,15,11,92,+8 Laps
18082,746,1960,Argentine Grand Prix,Moss,170,1,17,40,Suspension
20295,746,1960,Argentine Grand Prix,Moss,170,8,3,80,Finished


**Result:** in each pair the same driver has two different results in the same race. For example:
- **Moss, 1960 Argentine GP:** one row has grid 1, 40 laps, status `Suspension` and finishing place 17; the other has grid 8, 80 laps, status `Finished` and finishing place **3**.
- **Clark, 1964 US GP:** the same team and grid 1 in both rows, but 102 laps (place 7, `Out of fuel`) in one and 54 laps (place 12, `Injection`) in the other.
- **Greene, 1962 British GP:** two different teams, and both rows are `Withdrew` with 0 laps.

The project description explains that in the 1950s drivers could swap cars during a race, which gives a driver two rows. These are real records, not copies, so we cannot just delete one without a rule.

In [52]:
# What is different between the rows of a pair? (number of pairs, out of all repeated pairs)
podium = repeated.assign(podium=repeated["positionOrder"] <= 3).groupby(["raceId", "driverId"])["podium"]
differs = pd.Series({
    "team":                    (pairs["constructorId"].nunique() > 1).sum(),
    "grid position":           (pairs["grid"].nunique() > 1).sum(),
    "finishing place":         (pairs["positionOrder"].nunique() > 1).sum(),
    "laps":                    (pairs["laps"].nunique() > 1).sum(),
    "reason the race ended":   (pairs["statusId"].nunique() > 1).sum(),
    "podium (yes for one row, no for the other)": (podium.nunique() > 1).sum(),
}, name=f"pairs where it differs (out of {pairs.ngroups})")
differs.to_frame()

,pairs where it differs (out of 85)
team,16
grid position,81
finishing place,85
laps,82
reason the race ended,79
"podium (yes for one row, no for the other)",21


**Result:** in all 85 pairs the finishing place is different, and in most of them the grid position (81), the laps (82) and the reason the race ended (79) are different too. The team differs in only 16 pairs, so in most pairs both rows are for the same team.

Most important for us: in **21 pairs one row is a podium and the other is not**. The row we keep decides whether that driver counts as a podium finisher in that race, so the rule for choosing the row matters. This is decision B1 in the decisions file.

### Applying the rule: keep the car the driver started in
**Rule (decisions B1 and B2):** for every driver-race pair, keep the row with the **lowest `resultId`**, which is the car the driver started in. Grid and team come from that same row.

First we compare this rule with the other possible rules on the 85 repeated pairs, then we apply it. The raw tables in `text` are not changed: the result is a new table, `results_one_row`.

In [ ]:
# Compare the possible rules on the repeated driver-race pairs of `results`
res = text["results"].copy()
for col in ["resultId", "grid", "positionOrder", "laps"]:
    res[col] = res[col].astype(int)
res = res.sort_values("resultId")
key = ["raceId", "driverId"]
rows = res[res.duplicated(key, keep=False)].copy()             # the 176 rows of the 85 repeated pairs

# Does a row copy another driver's car? (same race, same grid and same finishing place as a DIFFERENT driver)
others = res[["raceId", "driverId", "grid", "positionOrder"]].rename(columns={"driverId": "other_driver"})
copy_ids = (rows.merge(others, on=["raceId", "grid", "positionOrder"])
                .query("driverId != other_driver")["resultId"].unique())
rows["copies_another_car"] = rows["resultId"].isin(copy_ids)

# The first row of each pair (lowest resultId) is the reference: the car the driver started in
first = rows.drop_duplicates(key, keep="first").set_index(key)

# What each rule would keep (one row per pair)
rules = {
    "first row (lowest resultId)":  rows.drop_duplicates(key, keep="first"),
    "best finishing place":         rows.sort_values(key + ["positionOrder", "resultId"]).drop_duplicates(key, keep="first"),
    "most laps":                    rows.sort_values(key + ["laps", "resultId"], ascending=[True, True, False, True]).drop_duplicates(key, keep="first"),
    "last row (highest resultId)":  rows.drop_duplicates(key, keep="last"),
}

table = {}
for name, kept in rules.items():
    kept = kept.set_index(key)
    first_of_kept = first.reindex(kept.index)                      # the first row of the same pairs, in the same order
    grid_gap = (kept["grid"] - first_of_kept["grid"]).abs()        # how far the kept grid is from the first row's grid
    table[name] = {
        "rows kept": len(kept),
        "kept rows that copy another car": int(kept["copies_another_car"].sum()),
        "pairs where the kept row is not the first row": int((kept["resultId"] != first_of_kept["resultId"]).sum()),
        "average grid gap vs first row (places)": round(grid_gap.mean(), 1),
        "podium rows kept": int((kept["positionOrder"] <= 3).sum()),
    }
table["drop the driver (remove all rows of the pair)"] = {
    "rows kept": 0, "kept rows that copy another car": 0, "pairs where the kept row is not the first row": "n/a",
    "average grid gap vs first row (places)": "n/a", "podium rows kept": 0}

# The brief's example rule "keep the row with a non-zero grid": how many pairs does it decide?
non_zero = rows.assign(non_zero=rows["grid"] > 0).groupby(key)["non_zero"].sum()
size = rows.groupby(key).size()
print("non-zero grid rule: pairs with exactly one non-zero row:", int((non_zero == 1).sum()),
      "| all rows non-zero:", int((non_zero == size).sum()), "| all rows zero:", int((non_zero == 0).sum()))
print("pairs whose rows have different car numbers:", int((rows.groupby(key)["number"].nunique() > 1).sum()), "of", rows.groupby(key).ngroups)
print("rows removed: first-row rule", len(rows) - len(first), "| drop the driver", len(rows))
print("later rows copying another car:", int(rows[rows.duplicated(key, keep="first")]["copies_another_car"].sum()),
      "of", int(rows.duplicated(key, keep="first").sum()), "| first rows:", int(first["copies_another_car"].sum()), "of", len(first))
# Show whole numbers without decimals (4.5 stays 4.5, 85.0 becomes 85)
pd.DataFrame(table).T.map(lambda v: f"{v:g}" if isinstance(v, (int, float)) else v)

### Why this rule, and why not the others
All numbers come from the table above (85 repeated pairs, 176 rows, 91 of them "later rows").

**The evidence behind "first row = the car he started in":**
- In all 85 pairs the car numbers of the rows are different, so the rows are different cars.
- Later rows copy another driver's grid and place in **84 of 91** cases; first rows do so in only **29 of 85**. The 29 is an upper bound: a first row also matches when a teammate later took over this driver's own car, which does not make it wrong.

| Option | What the numbers say | Verdict |
|---|---|---|
| **First row** (lowest `resultId`) | 29 of 85 kept rows match another car (upper bound). It is the reference, so its grid gap is 0. 91 rows removed. Costs 16 podium labels: 6 podium rows kept against 22 for the best-place rule, which is 0.06% of the 26,668 rows left. | **Chosen** |
| Best finishing place | **80 of 85** kept rows (94%) are another car's result. It picks a different row than the first row in **64 pairs**, and the kept grid is on average **4.5 places** away from the driver's own start over all pairs (about 6 places in the 64 pairs where the row differs). Keeps 22 podium rows. | Rejected: `grid` is our strongest feature and would be wrong for most of these drivers |
| Most laps | Same result as best place here: 80 of 85 other-car rows, 64 different rows, 4.5 places. The row with the most laps is usually the car he took over (e.g. Moss, 1960: 80 laps in the car he took over against 40 in his own). | Rejected |
| Last row (highest `resultId`) | 78 of 85 (92%) are another car's result, a different row than the first row in all 85 pairs, and a grid gap of 6.1 places. 17 podium rows. | Rejected |
| Drop the driver | Removes all **176** rows instead of 91, so 85 valid first rows are lost as well. | Rejected: throws away good data |
| "Keep the row with a non-zero grid" (the brief's example) | Decides **0 of the 85** pairs: 82 pairs have a non-zero grid in every row and 3 pairs have grid 0 in every row, so it never singles out one row. | Rejected: cannot choose |

**What we give up:** a driver who retired and then finished on the podium in the car he took over loses that podium label (16 labels). This is stated as a limitation in the report.

**Limits of the evidence:** there is no ground truth for "his own car", so the rule rests on the two points above. The 85 pairs are all in 1950 to 1964 (plus one in 1978), so they disappear anyway if we model only recent seasons.

In [53]:
# Keep the first row (lowest resultId) of every driver-race pair
raw_results = text["results"]

results_one_row = (raw_results.assign(_id=raw_results["resultId"].astype(int))   # resultId is still text: sort by its number
                              .sort_values("_id")
                              .drop_duplicates(["raceId", "driverId"], keep="first")
                              .drop(columns="_id")
                              .reset_index(drop=True))

# Before and after
podium_before = int((raw_results["positionOrder"].astype(int) <= 3).sum())
podium_after = int((results_one_row["positionOrder"].astype(int) <= 3).sum())
pd.Series({
    "rows before": len(raw_results),
    "rows after": len(results_one_row),
    "rows removed": len(raw_results) - len(results_one_row),
    "repeated driver-race pairs left": int(results_one_row.duplicated(["raceId", "driverId"]).sum()),
    "podium rows before": podium_before,
    "podium rows after": podium_after,
}, name="results").to_frame()

,results
rows before,26759
rows after,26668
rows removed,91
repeated driver-race pairs left,0
podium rows before,3397
podium rows after,3379


In [54]:
# Which rows were removed? Count per season
removed = raw_results[~raw_results["resultId"].isin(results_one_row["resultId"])]
season = removed["raceId"].map(text["races"].set_index("raceId")["year"]).astype(int)
season.value_counts().sort_index().rename("rows removed").rename_axis("year").to_frame()

,rows removed
year,
1950,4
1951,4
1952,3
1953,16
1954,17
1955,18
1956,12
1957,8
1958,3


**Result:** the rule removes **91 rows** (26,759 → 26,668) and **no driver-race pair is left with more than one row**. `results_one_row` now has exactly one row per driver per race. The removed rows are all from 1950 to 1964, plus one in 1978.

**Podium rows:** 3,397 before, 3,379 after, so 18 podium rows are removed. Of these, 16 are drivers who earned a podium in the car they took over (they lose that podium label), and 2 are the two rows of drivers who had two podium rows in the same race (Farina and Trintignant, 1955 Argentine GP).